**1.Data Profiling**
Combine the country files and check the number of rows and columns.

In [ ]:
from google.colab import files
uploaded = files.upload()


Saving categories.csv to categories.csv
Saving GBvideos.csv to GBvideos.csv
Saving generate_dataset.py to generate_dataset.py
Saving INvideos.csv to INvideos.csv
Saving README.md to README.md
Saving USvideos.csv to USvideos.csv


In [ ]:
import pandas as pd
import glob

files = glob.glob("/content/*videos.csv")
files = [f for f in files if not f.endswith("categories.csv")]

df = pd.concat(
    [pd.read_csv(f, encoding="latin1").assign(
        country=f.split("/")[-1].replace("videos.csv", "")
    ) for f in files],
    ignore_index=True
)

print("Rows and columns:", df.shape)
df.head()

Rows and columns: (46230, 17)


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
0,UcYiQY22yfG,17.14.11,Sunny Park - Cats (Lyric Video)!,QuietChallenge75,10,2017-11-14T05:57:11.000Z,Robots|Election,217915,14918.0,1363,662.0,https://i.ytimg.com/vi/UcYiQY22yfG/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,IN
1,EutcTEIciae,17.14.11,Perfect Pasta Unboxing?!,UltimateUnboxing65,1,2017-11-08T13:18:04.000Z,Paris|Bangalore|Winter,115014,2739.0,287,209.0,https://i.ytimg.com/vi/EutcTEIciae/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,IN
2,rvuZ5CsBpg8,17.14.11,Wild Budget Challenge!!,Brand NewSecret51,24,2017-11-14T18:16:59.000Z,Space,51028,1786.0,322,365.0,https://i.ytimg.com/vi/rvuZ5CsBpg8/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,IN
3,2yJGS3uxgMP,17.14.11,Quiet Chess Mystery,UltimateUnboxing18,23,2017-11-11T20:04:47.000Z,Budget|Festival|Minecraft|Guitar,143370,5224.0,305,571.0,https://i.ytimg.com/vi/2yJGS3uxgMP/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,IN
4,W5pdmLZu7GK,17.14.11,Quiet Festival Prank,HonestInterview42,24,2017-11-14T05:23:55.000Z,[none],390220,15079.0,947,1942.0,https://i.ytimg.com/vi/W5pdmLZu7GK/default.jpg,False,False,False,Watch this video. Subscribe for more content a...,IN


In [ ]:
#see Data typr & Missing value:

display(df.dtypes)

missing = df.isna().mean() * 100
display(missing[missing > 0].round(2).rename("Missing %"))

,0
video_id,object
trending_date,object
title,object
channel_title,object
category_id,int64
publish_time,object
tags,object
views,int64
likes,float64
dislikes,int64


,Missing %
likes,0.03
comment_count,0.08
description,3.51


In [ ]:
# Remove Duplicates:

before = len(df)

df = df.drop_duplicates().copy()

print("Duplicates removed:", before - len(df))
print("New shape:", df.shape)

Duplicates removed: 230
New shape: (46000, 17)


In [ ]:
#Category Lookup:

categories = pd.read_csv("/content/categories.csv")

display(categories.head())
print(categories.columns.tolist())

,category_id,category_name
0,1,Film & Animation
1,2,Autos & Vehicles
2,10,Music
3,15,Pets & Animals
4,17,Sports


['category_id', 'category_name']


In [ ]:
#Add Category Names:

df = df.merge(categories, on="category_id", how="left")

display(df[["category_id", "category_name"]].head())
print("Missing category names:", df["category_name"].isna().sum())

,category_id,category_name
0,10,Music
1,1,Film & Animation
2,24,Entertainment
3,23,Comedy
4,24,Entertainment


Missing category names: 0


In [ ]:
#Remove Unavailable Videos:

if "video_error_or_removed" in df.columns:
    removed = df["video_error_or_removed"].astype(str).str.lower().isin(["true", "1"])
    print("Records removed:", removed.sum())
    df = df[~removed].copy()
else:
    print("No removed-video flag column found.")

print("Final shape:", df.shape)

Records removed: 92
Final shape: (45908, 18)


**3.Interactive Chart**

This interactive scatter plot compares video views and likes.
Colours represent categories, and hovering over a point shows the video title.
A scatter plot suits these variables because views and likes are both numeric.

In [ ]:
# chart views vs Like:

import plotly.express as px

fig = px.scatter(
    df, x="views", y="likes",
    color="category_name",
    hover_name="title",
    title="Views vs Likes by Category"
)
fig.show()

**4.Interpretation**

Videos with more views generally have more likes, showing a positive relationship.
Most records cluster at lower values, while a few reach exceptionally high views and likes.
A scatter plot is suitable because it shows the relationship between two numeric variables.

Music has the highest average views across all three countries.
GB leads in Music with about 3.7 million average views, followed by US at 2.9 million and IN at 2.4 million.
IN leads in several other categories, including Comedy and Entertainment.
A grouped bar chart is suitable for comparing average views across categories and countries.

In [ ]:
#Average Views by Category and Country:

avg = df.groupby(["category_name", "country"])["views"].mean().reset_index()

fig = px.bar(
    avg, x="category_name", y="views",
    color="country", barmode="group",
    title="Average Views by Category and Country",
    labels={"category_name": "Category", "views": "Average Views"}
)
fig.show()